# Python - Panda Basics

### Creating a Data table (manually)
    

In [ ]:
import pandas as pd

In [ ]:
data = {
    "Student_name":["Cao","Victoria","Heesoo","Luke","Yeng"],
    "Student_ID": [555015,555016,555017,555018,555019],
    "Course_name":["CISC211","CISC191","CISC187","CISC192","CISC179"]
} 




In [ ]:
data


In [ ]:
type(data)

In [ ]:
data.values()

In [ ]:
# set index starting from 1
df = pd.DataFrame(data, index=[1, 2, 3, 4, 5])

# check result
df

### importing csv file

In [ ]:
# ignores or handles error 
florida = pd.read_csv("florida_with_ocean_dist.csv", on_bad_lines='skip', engine='python')
florida.index = range(1, len(florida) + 1)
florida.head(10) # shows 10 rows from the top

In [ ]:
florida.info

In [ ]:
florida.plot()


---

## Florida Dataset Wrangling Practice 


 `rename`, `reset_index`, `sort_values`, `drop_duplicates`, `dropna`, column subset, row subset, `query`, `groupby`, `pivot_table`, `concat`, `melt`,


In [1]:
# 1) Load CSV again as a raw backup
# Keep the original dataframe untouched, then clean a copy.

import pandas as pd
import numpy as np

florida_raw = pd.read_csv(
    "florida_with_ocean_dist.csv",
    on_bad_lines="skip",
    engine="python"
)

print("Rows, columns:", florida_raw.shape)
florida_raw.head()

FileNotFoundError: [Errno 2] No such file or directory: 'florida_with_ocean_dist.csv'

In [ ]:
# 2) Quick inspection
# Cheat sheet idea: understand the dataframe before changing it.

print("Column names:")
print(list(florida_raw.columns))

print("\nMissing values by column:")
display(florida_raw.isna().sum().sort_values(ascending=False))

print("\nData types:")
display(florida_raw.dtypes)

In [ ]:
# 3) Rename columns into clean GitHub-friendly snake_case
# Cheat sheet: df.rename(columns={...})

florida = florida_raw.copy()

florida.columns = (
    florida.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

# If the ocean-distance column has a slightly different name, normalize it.
for col in florida.columns:
    if "ocean" in col and ("dist" in col or "distance" in col):
        florida = florida.rename(columns={col: "ocean_distance_miles"})

print(list(florida.columns))
florida.head()

In [ ]:
# 4) Remove duplicate rows and reset index
# Cheat sheet: drop duplicates + reset_index

before_rows = len(florida)
florida = florida.drop_duplicates().reset_index(drop=True)
after_rows = len(florida)

print(f"Removed {before_rows - after_rows} duplicate rows.")
florida.head()

In [ ]:
# 5) Convert numeric-looking text columns into numbers
# Example: "1,234" -> 1234, "$250" -> 250, "14%" -> 14

for col in florida.columns:
    if florida[col].dtype == "object":
        cleaned = (
            florida[col]
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.replace("$", "", regex=False)
            .str.replace("%", "", regex=False)
            .str.strip()
        )
        numeric_version = pd.to_numeric(cleaned, errors="coerce")
        
        # Convert only if most non-empty values are numeric.
        non_empty = cleaned.replace("nan", np.nan).notna().sum()
        numeric_count = numeric_version.notna().sum()
        if non_empty > 0 and numeric_count / non_empty >= 0.70:
            florida[col] = numeric_version

florida.dtypes

In [ ]:
# 6) Subset useful columns
# Cheat sheet: df[["column1", "column2"]]
# This keeps common location + ocean-distance columns if they exist.

preferred_columns = [
    "name", "city", "county", "state", "latitude", "longitude",
    "lat", "lon", "lng", "population", "ocean_distance_miles"
]

columns_to_show = [col for col in preferred_columns if col in florida.columns]

if columns_to_show:
    display(florida[columns_to_show].head(10))
else:
    display(florida.head(10))

In [ ]:
# 7) Sort rows by ocean distance
# Cheat sheet: df.sort_values("column")

if "ocean_distance_miles" in florida.columns:
    florida_sorted = florida.sort_values("ocean_distance_miles", ascending=True).reset_index(drop=True)
    display(florida_sorted.head(10))
else:
    florida_sorted = florida.copy()
    print("No ocean_distance_miles column found. Check the column names above.")

In [ ]:
# 8) Subset observations / rows
# Cheat sheet: df[df.column > value], df.drop_duplicates(), df.sample()

if "ocean_distance_miles" in florida.columns:
    near_ocean = florida[florida["ocean_distance_miles"].between(0, 10, inclusive="both")]
    far_from_ocean = florida[florida["ocean_distance_miles"] > 50]

    print("Within 10 miles of ocean:", len(near_ocean))
    print("More than 50 miles from ocean:", len(far_from_ocean))
    display(near_ocean.head(10))
else:
    print("No ocean distance column available for row filtering.")

In [ ]:
# 9) Use query() for cleaner filtering
# Cheat sheet: df.query("column > value")

if "ocean_distance_miles" in florida.columns:
    near_ocean_query = florida.query("ocean_distance_miles <= 10")
    display(near_ocean_query.head(10))
else:
    print("No ocean_distance_miles column found.")

In [ ]:
# 10) Create a new categorical column
# This makes the dataset easier to summarize and visualize.

if "ocean_distance_miles" in florida.columns:
    florida["ocean_distance_group"] = pd.cut(
        florida["ocean_distance_miles"],
        bins=[-0.01, 5, 10, 25, 50, float("inf")],
        labels=["0-5 miles", "5-10 miles", "10-25 miles", "25-50 miles", "50+ miles"]
    )

    florida[["ocean_distance_miles", "ocean_distance_group"]].head(10)
else:
    print("No ocean_distance_miles column found.")

In [ ]:
# 11) Group and summarize
# Cheat sheet idea: reshape/summarize the data after cleaning.

if "ocean_distance_miles" in florida.columns:
    if "county" in florida.columns:
        county_summary = (
            florida
            .groupby("county", dropna=False)
            .agg(
                location_count=("ocean_distance_miles", "count"),
                avg_ocean_distance=("ocean_distance_miles", "mean"),
                min_ocean_distance=("ocean_distance_miles", "min"),
                max_ocean_distance=("ocean_distance_miles", "max")
            )
            .reset_index()
            .sort_values("avg_ocean_distance")
        )
        display(county_summary.head(15))
    else:
        ocean_summary = florida["ocean_distance_miles"].describe().to_frame().T
        display(ocean_summary)
else:
    print("No ocean_distance_miles column found.")

In [ ]:
# 12) Pivot table example
# Cheat sheet: df.pivot(...) / pivot_table(...)

if {"county", "ocean_distance_group"}.issubset(florida.columns):
    ocean_group_by_county = pd.pivot_table(
        florida,
        index="county",
        columns="ocean_distance_group",
        values="ocean_distance_miles",
        aggfunc="count",
        fill_value=0
    ).reset_index()

    display(ocean_group_by_county.head(15))
else:
    print("Need both county and ocean_distance_group columns for this pivot table.")

In [ ]:
# 13) Melt example: wide format -> long format
# Cheat sheet: pd.melt(df)

if "county_summary" in globals():
    county_summary_long = pd.melt(
        county_summary,
        id_vars="county",
        value_vars=["location_count", "avg_ocean_distance", "min_ocean_distance", "max_ocean_distance"],
        var_name="metric",
        value_name="value"
    )
    display(county_summary_long.head(20))
else:
    print("county_summary was not created because county/ocean distance columns were unavailable.")

In [ ]:
# 14) Concat example
# Cheat sheet: pd.concat([df1, df2])

if "ocean_distance_miles" in florida.columns:
    closest_5 = florida.sort_values("ocean_distance_miles", ascending=True).head(5)
    farthest_5 = florida.sort_values("ocean_distance_miles", ascending=False).head(5)

    ocean_extremes = pd.concat([closest_5, farthest_5], axis=0).reset_index(drop=True)
    display(ocean_extremes)
else:
    print("No ocean_distance_miles column found.")

In [ ]:
# 15) Save clean files for GitHub
# These CSV files can be uploaded to your GitHub repository.

florida.to_csv("florida_cleaned.csv", index=False)
print("Saved: florida_cleaned.csv")

if "county_summary" in globals():
    county_summary.to_csv("florida_county_summary.csv", index=False)
    print("Saved: florida_county_summary.csv")

if "ocean_group_by_county" in globals():
    ocean_group_by_county.to_csv("florida_ocean_group_by_county.csv", index=False)
    print("Saved: florida_ocean_group_by_county.csv")